In [88]:
from pathlib import Path
 
import cv2
import numpy as np
from PIL import Image
from typing import List

In [ ]:
_CROP_SIZE = 100
 
 
def extract_card_symbol_crops(
    image_rgb: np.ndarray,
    hsv_lower: List[np.ndarray],
    hsv_upper: List[np.ndarray],
    output_dir: Path | str | None = None,
    output_prefix: str = "crop",
) -> list[np.ndarray]:
    """
    Find every card of the given color (based on one or more min and max HSV thresholds) and return its symbol-corner crops.
    Optionally save the images to a directory (if needed for future training or testing).
    """
    # HSV color mask. This will change depending on which colored cards we are looking for
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    mask = np.zeros(image_rgb.shape[:2], dtype=np.uint8)
    for lo, hi in zip(hsv_lower, hsv_upper):
        mask = cv2.bitwise_or(mask, cv2.inRange(hsv, lo, hi))

 
    # A bit of morphology to clean thing. Open removes speckle, close stitches tiny gaps.
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
 
    # Find each colored region and try to fit a card swoosh to it.
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
 
    crops: list[np.ndarray] = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 1500:
            continue
 
        # Minimum enclosing triangle.
        try:
            tri_area, triangle = cv2.minEnclosingTriangle(cnt)
        except cv2.error:
            continue
        if triangle is None or tri_area <= 0:
            continue
 
        vertices = triangle.reshape(3, 2)
        contour_area = float(cv2.contourArea(cnt))
 
        # Sort triangle edges longest-first to identify the hypotenuse.
        edges = [(i, (i + 1) % 3,
                  float(np.linalg.norm(vertices[i] - vertices[(i + 1) % 3])))
                 for i in range(3)]
        edges.sort(key=lambda e: -e[2])
        long_len = edges[0][2]
        short_len = edges[2][2]
        ratio = long_len / short_len if short_len > 0 else 0.0
 
        # Card-shape filter. Reject if any basic threshold fails, OR if both
        # digit-reject thresholds (long edge and ratio) say it looks like the
        # big colored digit printed on the card center.
        if (long_len < 320
                or ratio < 1.4
                or contour_area < 13000
                or (long_len < 360 and ratio < 1.8)):
            continue
 
        long_i, long_j = edges[0][0], edges[0][1]
        tip_idx = ({0, 1, 2} - {long_i, long_j}).pop()
        tip = vertices[tip_idx]
        base_a = vertices[long_i]
        base_b = vertices[long_j]
 
        # Build the crop axes from the triangle's two legs. The longer
        # leg roughly aligns with the card's long axis, the perpendicular to
        # it (pointing toward the other leg) aligns with the short axis.
        leg_a = base_a - tip
        leg_b = base_b - tip
        if np.linalg.norm(leg_a) >= np.linalg.norm(leg_b):
            primary, other = leg_a, leg_b
        else:
            primary, other = leg_b, leg_a
        axis1 = primary / np.linalg.norm(primary)
        axis2 = np.array([-axis1[1], axis1[0]])
        if np.dot(axis2, other) < 0:
            axis2 = -axis2
 
        # Affine warp to get corner as a square (triangle is never a perfect right triangle)
        src = np.float32([
            tip,
            tip + axis1 * _CROP_SIZE,
            tip + axis2 * _CROP_SIZE,
        ])
        dst = np.float32([
            [0, 0],
            [_CROP_SIZE, 0],
            [0, _CROP_SIZE],
        ])
        M = cv2.getAffineTransform(src, dst)
        crops.append(cv2.warpAffine(image_rgb, M, (_CROP_SIZE, _CROP_SIZE)))
 
    # Optional save.
    if output_dir is not None:
        out_path = Path(output_dir)
        out_path.mkdir(parents=True, exist_ok=True)
        for idx, crop in enumerate(crops):
            cv2.imwrite(
                str(out_path / f"{output_prefix}_{idx}.png"),
                cv2.cvtColor(crop, cv2.COLOR_RGB2BGR),
            )
 
    return crops


In [90]:
def get_yellow_symbols(
        image: np.ndarray,
        output_dir: Path | str | None = None,
        output_prefix: str = "yellow_crop",
    ):
    return extract_card_symbol_crops(
        image_rgb=image,
        hsv_lower=[np.array([20, 120, 200], dtype=np.uint8)],
        hsv_upper=[np.array([32, 255, 255], dtype=np.uint8)],
        output_dir=output_dir,
        output_prefix = output_prefix,
    )

In [91]:
def get_blue_symbols(
        image: np.ndarray,
        output_dir: Path | str | None = None,
        output_prefix: str = "blue_crop",
    ):
    return extract_card_symbol_crops(
        image_rgb=image,
        hsv_lower=[np.array([92,  100, 150], dtype=np.uint8)],
        hsv_upper=[np.array([108, 255, 255], dtype=np.uint8)],
        output_dir=output_dir,
        output_prefix = output_prefix,
    )

In [92]:
def get_green_symbols(
        image: np.ndarray,
        output_dir: Path | str | None = None,
        output_prefix: str = "green_crop",
    ):
    return extract_card_symbol_crops(
        image_rgb=image,
        hsv_lower=[np.array([45, 80, 130], dtype=np.uint8)],
        hsv_upper=[np.array([68, 255, 255], dtype=np.uint8)],
        output_dir=output_dir,
        output_prefix = output_prefix,
    )

In [93]:
def get_red_symbols(
        image: np.ndarray,
        output_dir: Path | str | None = None,
        output_prefix: str = "red_crop",
    ):
    return extract_card_symbol_crops(
        image_rgb=image,
        hsv_lower=[
            np.array([0,   155, 220], dtype=np.uint8),
            np.array([177, 155, 220], dtype=np.uint8)
        ],
        hsv_upper=
            [np.array([5,   255, 255], dtype=np.uint8),
             np.array([179, 255, 255], dtype=np.uint8)
        ],
        output_dir=output_dir,
        output_prefix = output_prefix,
    )

In [94]:
def get_black_symbols(
        image: np.ndarray,
        output_dir: Path | str | None = None,
        output_prefix: str = "black_crop",
    ):
    return extract_card_symbol_crops(
        image_rgb=image,
        hsv_lower=[np.array([0, 0, 0], dtype=np.uint8)],
        hsv_upper=[np.array([40, 255, 130], dtype=np.uint8)],
        output_dir=output_dir,
        output_prefix = output_prefix,
    )

In [95]:
for img_path in sorted(Path("../data/iapr-26-uno-vision-challenge/test_images").iterdir()):
    if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        continue
    img = np.array(Image.open(img_path).convert("RGB"))
    output_path = Path("./test_test_crops")
    yellow_crops = get_yellow_symbols(img, output_path, output_prefix=f"{img_path.stem}_yellow_crop")
    red_crops = get_red_symbols(img, output_path, output_prefix=f"{img_path.stem}_red_crop")
    blue_crops = get_blue_symbols(img, output_path, output_prefix=f"{img_path.stem}_blue_crop")
    green_crops = get_green_symbols(img, output_path, output_prefix=f"{img_path.stem}_green_crop")
    black_crops = get_black_symbols(img, output_path, output_prefix=f"{img_path.stem}_black_crop")
    print(f"{img_path.name}: {len(yellow_crops)} yellow crops, {len(red_crops)} red crops, {len(blue_crops)} blue crops, {len(green_crops)} green crops, {len(black_crops)} black crops")

L1000793.jpg: 4 yellow crops, 7 red crops, 2 blue crops, 6 green crops, 0 black crops
L1000794.jpg: 4 yellow crops, 4 red crops, 2 blue crops, 2 green crops, 0 black crops
L1000795.jpg: 4 yellow crops, 5 red crops, 2 blue crops, 3 green crops, 2 black crops
L1000796.jpg: 2 yellow crops, 2 red crops, 6 blue crops, 9 green crops, 2 black crops
L1000797.jpg: 4 yellow crops, 2 red crops, 0 blue crops, 1 green crops, 0 black crops
L1000798.jpg: 0 yellow crops, 2 red crops, 2 blue crops, 6 green crops, 0 black crops
L1000799.jpg: 0 yellow crops, 4 red crops, 4 blue crops, 2 green crops, 0 black crops
L1000800.jpg: 0 yellow crops, 3 red crops, 0 blue crops, 5 green crops, 2 black crops
L1000801.jpg: 8 yellow crops, 0 red crops, 2 blue crops, 0 green crops, 0 black crops
L1000802.jpg: 0 yellow crops, 6 red crops, 0 blue crops, 6 green crops, 2 black crops
L1000803.jpg: 2 yellow crops, 3 red crops, 0 blue crops, 6 green crops, 0 black crops
L1000804.jpg: 6 yellow crops, 2 red crops, 2 blue crop